# FINAL MODEL

### CNN v3

v1 -handrafted +layer +epochs&earlystopping +weightdecay

concat->linear endir

In [ ]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 1738
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def r2(y, p): return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

def metrics(log_pred, age):
    pred = np.exp(log_pred)
    return {'RMSE': np.sqrt(np.mean((pred - age) ** 2)), 'MAE': np.mean(np.abs(pred - age)), 'R2 (age)': r2(age, pred), 'R2 (log age)': r2(np.log(age), log_pred)}

data in da memory

In [ ]:
DATA = Path('eeg_starter/data')
FS, WIN_S, REGION_S, C = 64, 8, 300, 20
WIN, N_WIN = WIN_S * FS, REGION_S // WIN_S

def load_split(name):
    seg = torch.load(DATA / f'segments_h1k_{name}.pt')
    X = torch.stack([torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt'))[:, :N_WIN * WIN] .unflatten(1, (N_WIN, WIN)).transpose(0, 1) for s in seg])
    return X, np.array([s[3] for s in seg])

X_tr, age_tr = load_split('train')
X_va, age_va = load_split('val')
print(X_tr.shape, X_va.shape)

torch.Size([825, 37, 20, 512]) torch.Size([103, 37, 20, 512])


3rd times the charm ;)

In [ ]:
class CNNv3(nn.Module):
    def __init__(self, kernels=(5, 5, 5), filters=(16, 32, 64), pool=2):
        super().__init__()
        layers, f_in = [], 1
        for k, f in zip(kernels, filters):
            layers += [nn.Conv1d(C * f_in, C * f, k, groups=C), nn.ReLU()]
            f_in = f
        self.conv = nn.Sequential(*layers, nn.MaxPool1d(pool), nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.fc = nn.Linear(C * filters[-1], 1)

    def forward(self, x):
        return self.fc(self.conv(x)).squeeze(1)

multi-use run fall

0mægaaa

In [ ]:
DEFAULTS = dict(kernels=(5, 5, 5), filters=(16, 32, 64), pool=2, epochs=50, patience=10, batch=64, lr=1e-3, weight_decay=1e-4)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return np.array([model(x.float().to(DEVICE)).mean().item() for x in X])

def run(**overrides):
    cfg = {**DEFAULTS, **overrides}
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = CNNv3(cfg['kernels'], cfg['filters'], cfg['pool']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    loss_fn = nn.MSELoss()
    y = torch.tensor(np.log(age_tr), dtype=torch.float32)
    child = torch.arange(len(X_tr)).repeat_interleave(N_WIN)
    win = torch.arange(N_WIN).repeat(len(X_tr))

    history, best, wait = [], np.inf, 0
    for epoch in range(cfg['epochs']):
        model.train()
        for b in torch.randperm(len(child)).split(cfg['batch']):
            c, w = child[b], win[b]
            loss = loss_fn(model(X_tr[c, w].float().to(DEVICE)), y[c].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        history.append(metrics(predict(model, X_va), age_va))
        print(f"epoch {epoch + 1:2d}  val RMSE {history[-1]['RMSE']:.3f}  MAE {history[-1]['MAE']:.3f}")
        if history[-1]['RMSE'] < best:
            best, best_epoch, best_state, wait = history[-1]['RMSE'], epoch + 1, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= cfg['patience']:
                break
    model.load_state_dict(best_state)

    rf = sum(k - 1 for k in cfg['kernels']) + cfg['pool'] # receptive field
    summary = {**cfg, 'receptive field (s)': rf / FS, 'params': sum(p.numel() for p in model.parameters()), 'best epoch': best_epoch, **history[best_epoch - 1]}
    return model, summary, history

kernel tests:

In [8]:
KERNELS = [10, 15, 20]

runs = {}
for k in KERNELS:
    print(f'--- kernel size {k}')
    runs[k] = run(kernels=(k, k, k))

pd.DataFrame([s for _, s, _ in runs.values()]).set_index('kernels')[['receptive field (s)', 'params', 'best epoch', 'RMSE', 'MAE']].round(3)

--- kernel size 10
epoch  1  val RMSE 2.078  MAE 1.284
epoch  2  val RMSE 1.231  MAE 0.858
epoch  3  val RMSE 1.492  MAE 0.926
epoch  4  val RMSE 1.423  MAE 0.883
epoch  5  val RMSE 1.232  MAE 0.847
epoch  6  val RMSE 1.179  MAE 0.726
epoch  7  val RMSE 1.173  MAE 0.805
epoch  8  val RMSE 1.071  MAE 0.716
epoch  9  val RMSE 1.198  MAE 0.832
epoch 10  val RMSE 1.168  MAE 0.733
epoch 11  val RMSE 1.079  MAE 0.732
epoch 12  val RMSE 1.008  MAE 0.636
epoch 13  val RMSE 0.965  MAE 0.655
epoch 14  val RMSE 1.000  MAE 0.659
epoch 15  val RMSE 0.972  MAE 0.675
epoch 16  val RMSE 1.118  MAE 0.696
epoch 17  val RMSE 1.270  MAE 0.743
epoch 18  val RMSE 1.009  MAE 0.702
epoch 19  val RMSE 0.944  MAE 0.620
epoch 20  val RMSE 1.061  MAE 0.643
epoch 21  val RMSE 1.027  MAE 0.712
epoch 22  val RMSE 1.330  MAE 0.792
epoch 23  val RMSE 1.030  MAE 0.665
epoch 24  val RMSE 1.192  MAE 0.718
epoch 25  val RMSE 1.110  MAE 0.669
epoch 26  val RMSE 0.997  MAE 0.680
epoch 27  val RMSE 1.203  MAE 0.743
epoch 28 

,receptive field (s),params,best epoch,RMSE,MAE
kernels,,,,,
"(10, 10, 10)",0.453,518721,19,0.944,0.620
"(15, 15, 15)",0.688,776321,20,0.912,0.598
"(20, 20, 20)",0.922,1033921,13,0.978,0.632


patience tests and 5x5 kernel tests accidentally overwritten. 5x5 was obviously too small so it was removed. 

15x15 <-

overlap?

overfit??????

shrinking kernel sizes?

In [9]:
for k, (model, s, _) in runs.items():
      tr = metrics(predict(model, X_tr), age_tr)
      print(k, 'train RMSE', round(float(tr['RMSE']), 3), 'MAE', round(float(tr['MAE']), 3), '| val RMSE', round(float(s['RMSE']), 3))

10 train RMSE 1.039 MAE 0.656 | val RMSE 0.944
15 train RMSE 1.039 MAE 0.644 | val RMSE 0.912
20 train RMSE 1.107 MAE 0.711 | val RMSE 0.978


not overfitting

New run function to log shit better:

In [ ]:
import time

DEFAULTS = {**DEFAULTS, 'train_metrics': True, 'schedule': None, 'lr_factor': 0.5, 'lr_patience': 2, 'min_lr': 1e-5, 'select': 'val RMSE', 'deterministic': False}

def run(name, X_train=None, X_val=None, **overrides):
    cfg = {**DEFAULTS, **overrides}
    X_train = X_tr if X_train is None else X_train
    X_val = X_va if X_val is None else X_val
    n_win = X_train.shape[1]
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    torch.backends.cudnn.deterministic = cfg['deterministic'] # why tf is this needed????? wat sa fak apple maps
    torch.backends.cudnn.benchmark = False
    model = CNNv3(cfg['kernels'], cfg['filters'], cfg['pool']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched = None
    if cfg['schedule'] == 'plateau':
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=cfg['lr_factor'], patience=cfg['lr_patience'], min_lr=cfg['min_lr'])
    elif cfg['schedule'] == 'cosine':
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'], eta_min=cfg['min_lr'])
    loss_fn = nn.MSELoss()
    y = torch.tensor(np.log(age_tr), dtype=torch.float32)
    child = torch.arange(len(X_train)).repeat_interleave(n_win)
    win = torch.arange(n_win).repeat(len(X_train))

    res = dict(name=name, config=cfg, n_windows=n_win, params=sum(p.numel() for p in model.parameters()), receptive_field_s=(sum(k - 1 for k in cfg['kernels']) + cfg['pool']) / FS, history=[], best_epoch=None, age_train=age_tr, age_val=age_va, pred_train=None, pred_val=None)
    Path('runs').mkdir(exist_ok=True)
    best, wait = np.inf, 0
    for epoch in range(cfg['epochs']):
        t0 = time.time()
        model.train(); total = 0
        for b in torch.randperm(len(child)).split(cfg['batch']):
            c, w = child[b], win[b]
            loss = loss_fn(model(X_train[c, w].float().to(DEVICE)), y[c].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.detach() * len(b)
        p_va = predict(model, X_val)
        p_tr = predict(model, X_train) if cfg['train_metrics'] else None
        h = {'epoch': epoch + 1, 'lr': opt.param_groups[0]['lr'], 'train loss': total.item() / len(child)}
        if p_tr is not None:
            h.update({f'train {k}': float(v) for k, v in metrics(p_tr, age_tr).items()})
            h['train RMSE (log)'] = float(np.sqrt(np.mean((p_tr - np.log(age_tr)) ** 2)))
        h.update({f'val {k}': float(v) for k, v in metrics(p_va, age_va).items()})
        h['val RMSE (log)'] = float(np.sqrt(np.mean((p_va - np.log(age_va)) ** 2)))
        h['seconds'] = time.time() - t0
        res['history'].append(h)
        improved = h[cfg['select']] < best
        print(f"epoch {h['epoch']:2d}  lr {h['lr']:.1e}  loss {h['train loss']:.3f}  " 
              + (f"train RMSE {h['train RMSE']:.3f}  " if p_tr is not None else '')
              + f"val RMSE {h['val RMSE']:.3f}  MAE {h['val MAE']:.3f}  RMSE (log) {h['val RMSE (log)']:.3f}  ({h['seconds']:.0f} s)"
              + ('  * best' if improved else ''))
        if improved:
            best, wait = h[cfg['select']], 0
            res.update(best_epoch=epoch + 1, state_dict={k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, pred_val=np.exp(p_va), pred_train=None if p_tr is None else np.exp(p_tr))
        else:
            wait += 1
        if cfg['schedule'] == 'plateau':
            sched.step(h[cfg['select']])
        elif sched is not None:
            sched.step()
        torch.save(res, f'runs/{name}.pt')
        if wait >= cfg['patience']:
            break

    model.load_state_dict(res['state_dict'])
    if res['pred_train'] is None:
        res['pred_train'] = np.exp(predict(model, X_train))
        torch.save(res, f'runs/{name}.pt')
    return {**res, 'model': model}

def summary(res):
    row = dict(res['history'][res['best_epoch'] - 1])
    row.pop('epoch')
    if 'lr' in row:
        row['lr'] = f"{row['lr']:.1e}"
    return {'name': res['name'], 'best epoch': res['best_epoch'], 'windows': res['n_windows'], 'params': res['params'], 'receptive field (s)': res['receptive_field_s'], **row}

def load_run(name):
    res = torch.load(f'runs/{name}.pt', weights_only=False)
    cfg = res['config']
    model = CNNv3(cfg['kernels'], cfg['filters'], cfg['pool']).to(DEVICE)
    model.load_state_dict(res['state_dict'])
    return {**res, 'model': model}

FULL 900S

In [ ]:
N_FULL = 900 // WIN_S 

def load_full(name):
    seg = torch.load(DATA / f'segments_h1k_{name}.pt')
    X = torch.empty(len(seg), N_FULL, C, WIN, dtype=torch.float16)
    for i, s in enumerate(seg):
        X[i] = torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt'))[:, :N_FULL * WIN].unflatten(1, (N_FULL, WIN)).transpose(0, 1)
    return X

X_tr_full, X_va_full = load_full('train'), load_full('val')
res_full = run('k15_full', X_train=X_tr_full, X_val=X_va_full, kernels=(15, 15, 15))
pd.DataFrame([summary(res_full)]).set_index('name')[['best epoch', 'train RMSE', 'train MAE', 'val RMSE', 'val MAE']].round(3)

epoch  1  loss 0.989  train RMSE 1.624  val RMSE 1.578  MAE 0.981  (53 s)
epoch  2  loss 0.267  train RMSE 1.776  val RMSE 1.493  MAE 0.993  (52 s)
epoch  3  loss 0.200  train RMSE 1.267  val RMSE 1.151  MAE 0.714  (52 s)
epoch  4  loss 0.173  train RMSE 1.211  val RMSE 1.118  MAE 0.661  (53 s)
epoch  5  loss 0.158  train RMSE 1.122  val RMSE 1.118  MAE 0.689  (53 s)
epoch  6  loss 0.148  train RMSE 1.111  val RMSE 1.080  MAE 0.645  (53 s)
epoch  7  loss 0.146  train RMSE 1.322  val RMSE 1.249  MAE 0.731  (53 s)
epoch  8  loss 0.139  train RMSE 1.011  val RMSE 0.985  MAE 0.577  (54 s)
epoch  9  loss 0.134  train RMSE 1.017  val RMSE 0.962  MAE 0.568  (54 s)
epoch 10  loss 0.132  train RMSE 1.381  val RMSE 1.307  MAE 0.776  (54 s)
epoch 11  loss 0.128  train RMSE 1.442  val RMSE 1.364  MAE 0.785  (55 s)
epoch 12  loss 0.127  train RMSE 1.117  val RMSE 1.048  MAE 0.629  (55 s)
epoch 13  loss 0.126  train RMSE 0.934  val RMSE 0.929  MAE 0.594  (55 s)
epoch 14  loss 0.130  train RMSE 1.038

,best epoch,train RMSE,train MAE,val RMSE,val MAE
name,,,,,
k15_full,14,1.038,0.659,0.876,0.574


using full dataset shows small difference val RMSE 0.876 vs 300s run's 0.912 - enough to not be noise?

schedule test because last run was jumpy:

In [12]:
res_plateau = run('k15_300s_plateau', kernels=(15, 15, 15), schedule='plateau')
pd.DataFrame([summary(res_plateau)]).set_index('name')[['best epoch', 'train RMSE', 'train MAE', 'val RMSE', 'val MAE']].round(3)

epoch  1  lr 1.0e-03  loss 1.437  train RMSE 3.291  val RMSE 2.055  MAE 1.520  (18 s)
epoch  2  lr 1.0e-03  loss 0.403  train RMSE 1.402  val RMSE 1.352  MAE 0.852  (19 s)
epoch  3  lr 1.0e-03  loss 0.297  train RMSE 1.345  val RMSE 1.203  MAE 0.805  (18 s)
epoch  4  lr 1.0e-03  loss 0.238  train RMSE 1.713  val RMSE 1.392  MAE 1.010  (18 s)
epoch  5  lr 1.0e-03  loss 0.199  train RMSE 1.384  val RMSE 1.231  MAE 0.760  (18 s)
epoch  6  lr 1.0e-03  loss 0.184  train RMSE 1.155  val RMSE 1.201  MAE 0.754  (17 s)
epoch  7  lr 1.0e-03  loss 0.162  train RMSE 1.252  val RMSE 0.982  MAE 0.665  (17 s)
epoch  8  lr 1.0e-03  loss 0.144  train RMSE 1.069  val RMSE 1.061  MAE 0.694  (17 s)
epoch  9  lr 1.0e-03  loss 0.149  train RMSE 1.076  val RMSE 0.991  MAE 0.667  (17 s)
epoch 10  lr 1.0e-03  loss 0.136  train RMSE 1.213  val RMSE 1.156  MAE 0.725  (18 s)
epoch 11  lr 5.0e-04  loss 0.120  train RMSE 1.337  val RMSE 1.263  MAE 0.768  (18 s)
epoch 12  lr 5.0e-04  loss 0.116  train RMSE 1.057  va

,best epoch,train RMSE,train MAE,val RMSE,val MAE
name,,,,,
k15_300s_plateau,28,0.951,0.614,0.855,0.582


W

Overlap test:

In [ ]:
assert 'select' in DEFAULTS, 'Rerun the logging run() cell first (the version with select)'

STRIDE_S = 4 # overlap
STRIDE = STRIDE_S * FS
N_SAMPLES = (900 // WIN_S) * WIN

def load_overlapping(name):
    seg = torch.load(DATA / f'segments_h1k_{name}.pt')
    S = torch.empty(len(seg), C, N_SAMPLES, dtype=torch.float16)
    for i, s in enumerate(seg):
        S[i] = torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt'))[:, :N_SAMPLES]
    return S.unfold(2, WIN, STRIDE).transpose(1, 2)

X_tr_full = X_va_full = None # RAAAM
X_tr_ov, X_va_ov = load_overlapping('train'), load_overlapping('val')
print(X_tr_ov.shape, X_va_ov.shape)

torch.Size([825, 223, 20, 512]) torch.Size([103, 223, 20, 512])


In [ ]:
res_ov = run('k15_full_overlap', X_train=X_tr_ov, X_val=X_va_ov, kernels=(15, 15, 15), schedule='plateau', select='val RMSE', epochs=100, deterministic=True)
pd.DataFrame([summary(res_ov)]).set_index('name')[['best epoch', 'train RMSE', 'train MAE', 'val RMSE', 'val MAE']].round(3)

epoch  1  lr 1.0e-03  loss 0.614  train RMSE 2.101  val RMSE 1.993  MAE 1.185  RMSE (log) 0.438  (111 s)  * best


epoch  2  lr 1.0e-03  loss 0.185  train RMSE 1.194  val RMSE 1.044  MAE 0.716  RMSE (log) 0.373  (111 s)  * best


epoch  3  lr 1.0e-03  loss 0.151  train RMSE 1.418  val RMSE 1.376  MAE 0.788  RMSE (log) 0.274  (111 s)


epoch  4  lr 1.0e-03  loss 0.138  train RMSE 1.047  val RMSE 0.969  MAE 0.586  RMSE (log) 0.251  (111 s)  * best


epoch  5  lr 1.0e-03  loss 0.132  train RMSE 1.230  val RMSE 1.194  MAE 0.717  RMSE (log) 0.258  (111 s)


epoch  6  lr 1.0e-03  loss 0.128  train RMSE 1.369  val RMSE 1.369  MAE 0.793  RMSE (log) 0.277  (110 s)


... (27 outputs left out)

epoch 34  lr 3.1e-05  loss 0.081  train RMSE 0.795  val RMSE 0.881  MAE 0.509  RMSE (log) 0.229  (111 s)


epoch 35  lr 3.1e-05  loss 0.081  train RMSE 0.809  val RMSE 0.879  MAE 0.507  RMSE (log) 0.222  (111 s)


epoch 36  lr 1.6e-05  loss 0.080  train RMSE 0.802  val RMSE 0.875  MAE 0.502  RMSE (log) 0.223  (111 s)


,best epoch,train RMSE,train MAE,val RMSE,val MAE
name,,,,,
k15_full_overlap,26,0.841,0.521,0.82,0.514


k15_full_overlap UNDISPUTED